In [1]:
from datetime import datetime
from html import escape

from z3 import And, Bool, BoolVal, Or, Solver


In [2]:
import pandas as pd


def symbolic_sqli_check(payload: str, status_code: int = 200, sql_error_tokens: list[str] | None = None) -> dict:
    """Classify one payload using deterministic symbolic rules + evidence trace."""
    sql_error_tokens = sql_error_tokens or []
    payload_lower = (payload or "").lower()

    has_quote_py = ("'" in payload) or ('"' in payload)
    has_comment_py = ("--" in payload) or ("/*" in payload) or ("*/" in payload)
    has_union_py = "union" in payload_lower
    has_tautology_py = (" or " in payload_lower) or (" and " in payload_lower)
    response_has_sql_error_py = len(sql_error_tokens) > 0
    response_5xx_py = (status_code or 0) >= 500

    has_quote = BoolVal(has_quote_py)
    has_comment = BoolVal(has_comment_py)
    has_union = BoolVal(has_union_py)
    has_tautology = BoolVal(has_tautology_py)
    response_has_sql_error = BoolVal(response_has_sql_error_py)
    response_5xx = BoolVal(response_5xx_py)

    looks_injection_payload = Or(has_quote, has_comment, has_union, has_tautology)
    impact_signal = Or(response_has_sql_error, response_5xx)

    vulnerable = Bool("vulnerable")
    s = Solver()
    s.add(
        vulnerable == Or(   
            And(looks_injection_payload, impact_signal),
            And(has_union, has_quote),
            And(has_tautology, has_quote),
        )
    )

    sat = str(s.check()) == "sat"
    confirmed = bool(s.model()[vulnerable]) if sat else False

    evidence_signals = []
    if has_quote_py:
        evidence_signals.append("payload_has_quote")
    if has_comment_py:
        evidence_signals.append("payload_has_sql_comment")
    if has_union_py:
        evidence_signals.append("payload_has_union")
    if has_tautology_py:
        evidence_signals.append("payload_has_boolean_logic")
    if response_has_sql_error_py:
        evidence_signals.append("response_has_sql_error_token")
    if response_5xx_py:
        evidence_signals.append("response_is_5xx")

    rule_hits = []
    if (has_quote_py or has_comment_py or has_union_py or has_tautology_py) and (response_has_sql_error_py or response_5xx_py):
        rule_hits.append("R1:injection_payload_AND_impact_signal")
    if has_union_py and has_quote_py:
        rule_hits.append("R2:union_AND_quote")
    if has_tautology_py and has_quote_py:
        rule_hits.append("R3:tautology_AND_quote")

    has_sqli_vuln = confirmed
    evidence_text = " | ".join(evidence_signals) if evidence_signals else "no_signal"
    rule_text = " | ".join(rule_hits) if rule_hits else "no_rule_hit"

    return {
        "payload": payload,
        "status_code": status_code,
        "sql_error_tokens": sql_error_tokens,
        "symbolic_confirmed": confirmed,
        "has_sqli_vuln": has_sqli_vuln,
        "has_quote": has_quote_py,
        "has_comment": has_comment_py,
        "has_union": has_union_py,
        "has_tautology": has_tautology_py,
        "response_has_sql_error": response_has_sql_error_py,
        "response_5xx": response_5xx_py,
        "evidence_signals": evidence_signals,
        "rule_hits": rule_hits,
        "evidence_text": evidence_text,
        "rule_text": rule_text,
    }


In [3]:
# Glossary: what each piece of evidence means

EVIDENCE_GLOSSARY = {
    "payload_has_quote": "Payload contains a quote character (' or \") — key SQL injection metachar.",
    "payload_has_sql_comment": "Payload contains SQL comment markers (-- or /* */) — used to truncate queries.",
    "payload_has_union": "Payload contains UNION keyword — classic multi-row injection technique.",
    "payload_has_boolean_logic": "Payload contains OR/AND keywords — used for boolean-based blind injection.",
    "response_has_sql_error_token": "Response contains SQL error text (e.g. syntax error, mysql, postgresql, sqlite) — indicates query execution failure.",
    "response_is_5xx": "Response status code is 5xx (server error) — may indicate an exception from injected SQL.",
}

RULE_GLOSSARY = {
    "R1:injection_payload_AND_impact_signal": "Payload has injection markers AND response shows SQL/server error — strong evidence of successful injection.",
    "R2:union_AND_quote": "Payload has both UNION and quotes — classic UNION-based injection pattern.",
    "R3:tautology_AND_quote": "Payload has both OR/AND logic and quotes — boolean-based injection pattern.",
}

def decode_evidence(evidence_signals: list[str]) -> list[str]:
    """Convert evidence codes to human-readable descriptions."""
    return [EVIDENCE_GLOSSARY.get(code, code) for code in evidence_signals]


def decode_rules(rule_hits: list[str]) -> list[str]:
    """Convert rule codes to human-readable descriptions."""
    return [RULE_GLOSSARY.get(code, code) for code in rule_hits]


print("Evidence Glossary:")
for code, desc in EVIDENCE_GLOSSARY.items():
    print(f"  {code}: {desc}")
print("\nRule Glossary:")
for code, desc in RULE_GLOSSARY.items():
    print(f"  {code}: {desc}")

Evidence Glossary:
  payload_has_quote: Payload contains a quote character (' or ") — key SQL injection metachar.
  payload_has_sql_comment: Payload contains SQL comment markers (-- or /* */) — used to truncate queries.
  payload_has_union: Payload contains UNION keyword — classic multi-row injection technique.
  payload_has_boolean_logic: Payload contains OR/AND keywords — used for boolean-based blind injection.
  response_has_sql_error_token: Response contains SQL error text (e.g. syntax error, mysql, postgresql, sqlite) — indicates query execution failure.
  response_is_5xx: Response status code is 5xx (server error) — may indicate an exception from injected SQL.

Rule Glossary:
  R1:injection_payload_AND_impact_signal: Payload has injection markers AND response shows SQL/server error — strong evidence of successful injection.
  R2:union_AND_quote: Payload has both UNION and quotes — classic UNION-based injection pattern.
  R3:tautology_AND_quote: Payload has both OR/AND logic and q

In [6]:
import json
import re
from pathlib import Path
from urllib.parse import parse_qs, urlparse, unquote_plus

# Resolve traffic path from either notebook dir or workspace root.
candidate_paths = [
    Path.cwd() / "rule_based_tool" / "zap_scan_traffic.json",
    Path.cwd().parent / "rule_based_tool" / "zap_scan_traffic.json",
]
traffic_path = next((p for p in candidate_paths if p.exists()), None)
if traffic_path is None:
    raise FileNotFoundError("Could not find rule_based_tool/zap_scan_traffic.json")

with open(traffic_path, "r", encoding="utf-8") as f:
    traffic = json.load(f)

print(f"Loaded {len(traffic)} traffic messages from: {traffic_path}")


def parse_request_meta(request_header: str) -> dict:
    lines = (request_header or "").splitlines()
    first_line = lines[0] if lines else ""
    parts = first_line.split()
    method = parts[0] if len(parts) >= 1 else ""
    target = parts[1] if len(parts) >= 2 else ""

    host = ""
    for line in lines[1:]:
        if line.lower().startswith("host:"):
            host = line.split(":", 1)[1].strip()
            break

    if target.startswith("http://") or target.startswith("https://"):
        full_url = target
    elif target.startswith("/") and host:
        full_url = f"http://{host}{target}"
    else:
        full_url = target

    parsed = urlparse(full_url if full_url else target)
    endpoint = parsed.path or "/"

    return {
        "method": method,
        "target": target,
        "host": host,
        "full_url": full_url,
        "endpoint": endpoint,
        "parsed": parsed,
    }


def response_sql_tokens(response_header: str, response_body: str) -> list[str]:
    haystack = ((response_header or "") + "\n" + (response_body or "")).lower()
    signatures = [
        "sql syntax",
        "sqlite",
        "mysql",
        "postgresql",
        "odbc",
        "unterminated",
        "syntax error",
        "warning",
    ]
    return [s for s in signatures if s in haystack]


def status_code_from_header(response_header: str) -> int:
    m = re.search(r"HTTP/\d\.\d\s+(\d{3})", response_header or "")
    return int(m.group(1)) if m else 0


rows = []
for msg in traffic:
    req = parse_request_meta(msg.get("requestHeader", ""))
    if not req["target"]:
        continue

    query_map = parse_qs(req["parsed"].query, keep_blank_values=True)
    body_map = parse_qs(msg.get("requestBody", ""), keep_blank_values=True)

    merged = {}
    merged.update(query_map)
    for k, v in body_map.items():
        merged.setdefault(k, []).extend(v)

    if not merged:
        continue

    sql_tokens = response_sql_tokens(msg.get("responseHeader", ""), msg.get("responseBody", ""))
    status = status_code_from_header(msg.get("responseHeader", ""))

    for param, values in merged.items():
        for raw_v in values:
            payload = unquote_plus(raw_v or "")
            verdict = symbolic_sqli_check(
                payload=payload,
                status_code=status,
                sql_error_tokens=sql_tokens,
            )
            rows.append(
                {
                    "message_id": msg.get("id"),
                    "method": req["method"],
                    "host": req["host"],
                    "url": req["full_url"],
                    "endpoint": req["endpoint"],
                    "param": param,
                    "payload": payload,
                    "param_value": f"{param}={payload}",
                    **{k: v for k, v in verdict.items() if k != "payload"},
                }
            )

zap_symbolic_df = pd.DataFrame(rows)
print(f"Evaluated payload values: {len(zap_symbolic_df)}")

if zap_symbolic_df.empty:
    print("No payload values extracted from traffic.")
else:
    summary_cols = [
        "message_id",
        "method",
        "url",
        "endpoint",
        "param",
        "payload",
        "symbolic_confirmed",
        "evidence_text",
        "rule_text",
    ]
    display(zap_symbolic_df[summary_cols].head(10))

    confirmed_df = zap_symbolic_df[zap_symbolic_df["symbolic_confirmed"] == True].copy()
    print(f"Symbolically confirmed rows: {len(confirmed_df)}")
    if not confirmed_df.empty:
        display(confirmed_df[summary_cols].head(10))

Loaded 196 traffic messages from: c:\Users\Administrator\Projects\neurosymbol_exp\rule_based_tool\zap_scan_traffic.json
Evaluated payload values: 378


,message_id,method,url,endpoint,param,payload,symbolic_confirmed,evidence_text,rule_text
0,10,GET,http://juice-shop.herokuapp.com/?id=%27&Submit...,/,id,',True,payload_has_quote | response_is_5xx,R1:injection_payload_AND_impact_signal
1,10,GET,http://juice-shop.herokuapp.com/?id=%27&Submit...,/,Submit,Submit,False,response_is_5xx,no_rule_hit
2,11,GET,http://juice-shop.herokuapp.com/?id=1%27&Submi...,/,id,1',True,payload_has_quote | response_is_5xx,R1:injection_payload_AND_impact_signal
3,11,GET,http://juice-shop.herokuapp.com/?id=1%27&Submi...,/,Submit,Submit,False,response_is_5xx,no_rule_hit
4,12,GET,http://juice-shop.herokuapp.com/?id=%22&Submit...,/,id,"""",True,payload_has_quote | response_is_5xx,R1:injection_payload_AND_impact_signal
5,12,GET,http://juice-shop.herokuapp.com/?id=%22&Submit...,/,Submit,Submit,False,response_is_5xx,no_rule_hit
6,13,GET,http://juice-shop.herokuapp.com/?id=1%22&Submi...,/,id,"1""",True,payload_has_quote | response_is_5xx,R1:injection_payload_AND_impact_signal
7,13,GET,http://juice-shop.herokuapp.com/?id=1%22&Submi...,/,Submit,Submit,False,response_is_5xx,no_rule_hit
8,14,GET,http://juice-shop.herokuapp.com/?id=%3B&Submit...,/,id,;,False,response_is_5xx,no_rule_hit
9,14,GET,http://juice-shop.herokuapp.com/?id=%3B&Submit...,/,Submit,Submit,False,response_is_5xx,no_rule_hit


Symbolically confirmed rows: 118


,message_id,method,url,endpoint,param,payload,symbolic_confirmed,evidence_text,rule_text
0,10,GET,http://juice-shop.herokuapp.com/?id=%27&Submit...,/,id,',True,payload_has_quote | response_is_5xx,R1:injection_payload_AND_impact_signal
2,11,GET,http://juice-shop.herokuapp.com/?id=1%27&Submi...,/,id,1',True,payload_has_quote | response_is_5xx,R1:injection_payload_AND_impact_signal
4,12,GET,http://juice-shop.herokuapp.com/?id=%22&Submit...,/,id,"""",True,payload_has_quote | response_is_5xx,R1:injection_payload_AND_impact_signal
6,13,GET,http://juice-shop.herokuapp.com/?id=1%22&Submi...,/,id,"1""",True,payload_has_quote | response_is_5xx,R1:injection_payload_AND_impact_signal
12,16,GET,http://juice-shop.herokuapp.com/?id=%27%28&Sub...,/,id,'(,True,payload_has_quote | response_is_5xx,R1:injection_payload_AND_impact_signal
14,17,GET,http://juice-shop.herokuapp.com/?id=1%27%28&Su...,/,id,1'(,True,payload_has_quote | response_is_5xx,R1:injection_payload_AND_impact_signal
28,24,GET,http://juice-shop.herokuapp.com/?id=%27%22&Sub...,/,id,"'""",True,payload_has_quote | response_is_5xx,R1:injection_payload_AND_impact_signal
30,25,GET,http://juice-shop.herokuapp.com/?id=1%27%22&Su...,/,id,"1'""",True,payload_has_quote | response_is_5xx,R1:injection_payload_AND_impact_signal
44,32,GET,http://juice-shop.herokuapp.com/?id=1+AND+1%3D...,/,id,1 AND 1=1 --,True,payload_has_sql_comment | payload_has_boolean_...,R1:injection_payload_AND_impact_signal
46,33,GET,http://juice-shop.herokuapp.com/?id=1+AND+1%3D...,/,id,1 AND 1=2 --,True,payload_has_sql_comment | payload_has_boolean_...,R1:injection_payload_AND_impact_signal


In [7]:
report_df = zap_symbolic_df.copy()
confirmed_df = report_df[report_df["symbolic_confirmed"] == True].copy()

# Decode evidence and rules into human-readable form
confirmed_df["evidence_plain"] = confirmed_df["evidence_signals"].apply(
    lambda signals: " | ".join(decode_evidence(signals)) if isinstance(signals, list) else ""
)
confirmed_df["rules_plain"] = confirmed_df["rule_hits"].apply(
    lambda rules: " | ".join(decode_rules(rules)) if isinstance(rules, list) else ""
)

# Add a site-level vulnerability tag so the exported report can answer
# whether a host/endpoint should be considered SQLi vulnerable overall.
site_vuln_df = (
    report_df.groupby(["host", "url"], dropna=False, as_index=False)["symbolic_confirmed"]
    .any()
    .rename(columns={"symbolic_confirmed": "site_has_sqli_vuln"})
)
report_df = report_df.merge(site_vuln_df, on=["host", "url"], how="left")
confirmed_df = report_df[report_df["symbolic_confirmed"] == True].copy()

# Keep the row-level alias in sync with the site-level tag for confirmed rows.
report_df["has_sqli_vuln"] = report_df["site_has_sqli_vuln"].fillna(False)
confirmed_df["has_sqli_vuln"] = confirmed_df["site_has_sqli_vuln"].fillna(False)

site_vuln_summary = (
    site_vuln_df.groupby("site_has_sqli_vuln").size().to_dict()
    if not site_vuln_df.empty else {}
)

total = len(report_df)
confirmed = len(confirmed_df)
rate = (confirmed / total * 100.0) if total else 0.0

top_params = (
    confirmed_df.groupby("param").size().sort_values(ascending=False).head(10).to_dict()
    if not confirmed_df.empty else {}
)

summary = {
    "generated_at": datetime.now().isoformat(timespec="seconds"),
    "total_rows": total,
    "confirmed_rows": confirmed,
    "confirmation_rate_pct": round(rate, 2),
    "top_confirmed_params": top_params,
    "site_vulnerability_summary": site_vuln_summary,
    "glossary_evidence": EVIDENCE_GLOSSARY,
    "glossary_rules": RULE_GLOSSARY,
}

out_dir = Path.cwd().parent / "rule_based_tool"
if not out_dir.exists():
    out_dir = Path.cwd() / "rule_based_tool"
out_dir.mkdir(parents=True, exist_ok=True)

summary_path = out_dir / "symbolic_report_summary.json"
rows_json_path = out_dir / "symbolic_report_rows.json"
rows_csv_path = out_dir / "symbolic_report_rows.csv"
confirmed_csv_path = out_dir / "symbolic_report_confirmed.csv"
html_path = out_dir / "symbolic_report.html"

summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")
report_df.to_json(rows_json_path, orient="records", indent=2)
report_df.to_csv(rows_csv_path, index=False)

export_cols = [
    "message_id",
    "method",
    "host",
    "url",
    "endpoint",
    "param",
    "payload",
    "param_value",
    "symbolic_confirmed",
    "has_sqli_vuln",
    "site_has_sqli_vuln",
    "status_code",
    "response_has_sql_error",
    "evidence_text",
    "evidence_plain",
    "rule_text",
    "rules_plain",
]
export_df = confirmed_df[[c for c in export_cols if c in confirmed_df.columns]].copy()
export_df.to_csv(confirmed_csv_path, index=False)

evidence_list_html = "".join([f"<li><b>{code}</b>: {desc}</li>" for code, desc in EVIDENCE_GLOSSARY.items()])
rules_list_html = "".join([f"<li><b>{code}</b>: {desc}</li>" for code, desc in RULE_GLOSSARY.items()])

rows_html = []
for _, r in confirmed_df.head(200).iterrows():
    msg_id = escape(str(r.get("message_id", "")))
    method = escape(str(r.get("method", "")))
    url = escape(str(r.get("url", "")))
    endpoint = escape(str(r.get("endpoint", "")))
    param = escape(str(r.get("param", "")))
    payload = escape(str(r.get("payload", "")))[:120]
    evidence_plain = escape(str(r.get("evidence_plain", "")))[:250]
    rules_plain = escape(str(r.get("rules_plain", "")))[:250]
    vuln_label = "yes" if bool(r.get("has_sqli_vuln", False)) else "no"
    rows_html.append(
        f"<tr><td>{msg_id}</td><td>{method}</td><td>{url}<br><small>{endpoint}</small></td><td>{param}</td><td><code>{payload}</code></td><td>{vuln_label}</td><td><small>{evidence_plain}</small></td><td><small>{rules_plain}</small></td></tr>"
    )

html_doc = f"""<!doctype html>
<html>
<head>
  <meta charset='utf-8'/>
  <title>Symbolic SQLi Evidence Report</title>
  <style>
    body {{ font-family: Segoe UI, Arial, sans-serif; margin: 20px; color: #1f2937; }}
    h2 {{ color: #1e40af; }}
    h3 {{ color: #1e40af; margin-top: 24px; }}
    .kpi {{ margin-right: 20px; display: inline-block; padding: 8px 12px; background: #f0f9ff; border-radius: 4px; }}
    table {{ border-collapse: collapse; width: 100%; font-size: 12px; margin-top: 12px; }}
    th, td {{ border: 1px solid #e5e7eb; padding: 6px; text-align: left; vertical-align: top; }}
    th {{ background: #f3f4f6; }}
    small {{ color: #6b7280; }}
    code {{ background: #f3f4f6; padding: 2px 4px; border-radius: 2px; }}
    .box {{ background: #fef3c7; border-left: 3px solid #f59e0b; padding: 10px; margin: 12px 0; }}
  </style>
</head>
<body>
  <h2>Symbolic SQLi Evidence Report</h2>
  <div class='kpi'><b>Total:</b> {total}</div>
  <div class='kpi'><b>Confirmed:</b> {confirmed}</div>
  <div class='kpi'><b>Rate:</b> {rate:.2f}%</div>

  <div class='box'>
    <h3>Evidence Signals Glossary</h3>
    <ul>{evidence_list_html}</ul>
  </div>

  <div class='box'>
    <h3>Rule Hits Glossary</h3>
    <ul>{rules_list_html}</ul>
  </div>

  <h3>Confirmed Findings</h3>
  <table>
    <tr><th>ID</th><th>Method</th><th>URL</th><th>Param</th><th>Payload</th><th>Vuln</th><th>Evidence</th><th>Rules</th></tr>
    {''.join(rows_html)}
  </table>
</body>
</html>"""

html_path.write_text(html_doc, encoding="utf-8")
print(f"Saved: {summary_path}")
print(f"Saved: {confirmed_csv_path}")
print(f"Saved: {html_path}")


Saved: c:\Users\Administrator\Projects\neurosymbol_exp\rule_based_tool\symbolic_report_summary.json
Saved: c:\Users\Administrator\Projects\neurosymbol_exp\rule_based_tool\symbolic_report_confirmed.csv
Saved: c:\Users\Administrator\Projects\neurosymbol_exp\rule_based_tool\symbolic_report.html
